In [1]:
# =============================================================================
# CELL 1: ALL CONFIGURATION, ASSUMPTIONS, BASELINES, INPUTS
# =============================================================================

# --- Granularity: 'q' = quarterly, 'm' = monthly, 'w' = weekly ---
# Use a list to run multiple granularities in one execution (e.g. ['m', 'q']).
# Weekly ('w') uses app_date and should be run separately from m/q.
granularities = ['w']         #  ['m', 'q'] #

# --- Date Range (inclusive) ---
START_DATE = '2026-01-01'
END_DATE = None  # None = auto-detect from today's date

# --- Query Control ---

run_every_query = True  # True = run SQL queries; False = use cached pickles

# --- Date Column per Granularity ---
# Q/M use book_date; W uses app_date (application_received_dtm)
DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

# --- LOBs to Process ---
LOBS = ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX']

# --- Rollup Groups (weighted-average aggregation of individual LOB results) ---
ROLLUP_GROUPS = {
    'Franchise Independent': ['AN', 'FLD', 'FRN', 'STG'],
    'nonKMX': ['AN', 'FRN', 'STG', 'FLD', 'ENT'],
    'POS': ['AN', 'FRN', 'STG', 'FLD', 'ENT', 'KMX'],
}

# --- Baselines (per individual LOB) ---
BASELINES = {
    'AN':  {'ltv': 1.94, 'new_recovery_unadjusted': 0.58, 'apr': 0.25}, #changed from 0.58
    'FRN': {'ltv': 1.94, 'new_recovery_unadjusted': 0.54, 'apr': 0.25}, #changed from 0.55
    'STG': {'ltv': 1.94, 'new_recovery_unadjusted': 0.60, 'apr': 0.25},
    'FLD': {'ltv': 1.45, 'new_recovery_unadjusted': 0.54, 'apr': 0.235}, #changed from 0.55
    'ENT': {'ltv': 1.45, 'new_recovery_unadjusted': 0.55, 'apr': 0.235},  
    'KMX': {'ltv': 1.59, 'new_recovery_unadjusted': 0.58, 'apr': 0.25},
}

# --- Model Parameters ---
MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,  #0.027 new sloping, for frn 3.1
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

# --- Excluded Vintages (per LOB) ---
# EXCLUDED_VINTAGES = {
#     'KMX': {'2022 M02', '2023 M11', '2022-05', '2022-06', '2022-07', '2022-08', '2022-09',
#             '2023-44', '2023-45', '2023-46', '2023-47', '2023-48'},
#     'AN':  {'2023-14', '2023-15'},
#     'FRN': {'2023-14', '2023-15'},
#     'STG': {'2023-14', '2023-15'},
#     'FLD': {'2023-14', '2023-15'},
#     'ENT': {'2023-14', '2023-15'},
# }
EXCLUDED_VINTAGES = {}

In [2]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
import datetime as dt
import re
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

# --- Derived values (do not modify) ---
# Normalize: support both legacy single-value `granularity` and new `granularities` list
if 'granularities' not in dir():
    granularities = [granularity]

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}

# Shared date_col: all granularities in a single run must share the same date column.
# m/q both use book_date; w uses app_date.
date_col = DATE_COL_MAP[granularities[0]]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

min_date_sql = f"'{START_DATE}'"

print(f"Granularities: {granularities}")
print(f"Date column: {date_col}")
for g in granularities:
    pf = PERIOD_FREQ_MAP[g]
    print(f"  {g}: {start_date.to_period(pf)} to {end_date.to_period(pf)}")
print(f"SQL min_date: {min_date_sql}")

Granularities: ['w']
Date column: app_date
  w: 2025-12-28/2026-01-03 to 2026-09-06/2026-09-12
SQL min_date: '2026-01-01'


In [3]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    """Fetch from SQL and cache to pickle. Reuse cache unless force_refresh=True
    or the pickle file is missing."""
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def smooth(series):
    averaged_series = pd.Series(index=series.index, dtype=float)
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = series.iloc[i-2:i+3].mean()
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    averaged_series.iloc[-1] = series.iloc[-1]
    return averaged_series


def weight_by_proceeds(metric, proceeds):
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def assign_period(df, col_name, freq):
    """Assign a pd.Period column from a date column."""
    dt_series = pd.to_datetime(df[col_name])
    return dt_series.dt.to_period(freq)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings (e.g. '2025 Q1', '2025 M01')."""
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)

In [4]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_kmx(ula_df, loss_scale=None, leave_out='None'):
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']

    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag))

    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag))

    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.05 * ula_df.high_pti_tier_1_flag
            + 0.1 * ula_df.high_pti_tier_2_flag
            + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag)



    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)



    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)
            * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag))

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Clip':
        mask = (ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag)
        ula_df.loc[mask, 'loss_multiplier'] = np.clip(ula_df.loc[mask, 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))

    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age

    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag

    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag

    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag

    if leave_out != 'Illinois':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.25 * ula_df.illinois_flag

    if leave_out != 'Mississippi':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.2 * ula_df.mississippi_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag)

    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag

    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag

    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))

        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag

    if leave_out != 'blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1

    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(
            ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df

In [5]:
# =============================================================================
# CELL 5: DATA FETCH (SQL + PICKLE)
# =============================================================================
# Each table has its own schema-tagged pickle under cache/. When
# run_every_query=False, each cached_sql call reuses the pickle if present
# and falls through to SQL if missing. Delete an individual pickle to force
# a selective refresh.

os.makedirs('cache', exist_ok=True)
force = run_every_query

# --- Model Scores ---
all_original_model_scores = cached_sql(
    '../queries/postmodern_ms_query.txt',
    '../../cache/ms_v1.pkl',
    sub_list=[('{min_book_date}', min_date_sql)],
    force_refresh=force,
)
print(f"Model scores fetched: {len(all_original_model_scores):,} records")

# --- ULA, DLA, New Recovery ---
# Share one Redshift connection across the three queries only when any of
# them actually needs to hit the DB; otherwise skip the connection entirely.
need_conn = force or not all(
    os.path.exists(p) for p in ('../../cache/ula_v1.pkl', '../../cache/dla_v1.pkl', '../../cache/new_recovery_v1.pkl')
)

if need_conn:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        ula_df_total = cached_sql(
            '../queries/vintage_level_ula_query.txt', '../../cache/ula_v1.pkl',
            sub_list=[('{min_book_date}', min_date_sql)], connection=conn, force_refresh=force,
        )
        print('ULA ready')

        dla_df = cached_sql(
            '../queries/new_dll_query.txt', '../../cache/dla_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('DLA ready')

        new_recovery = cached_sql(
            '../queries/new_recovery_queryt.txt', '../../cache/new_recovery_v1.pkl',
            connection=conn, force_refresh=force,
        )
        print('New recovery ready')
else:
    ula_df_total = get_pickle('../../cache/ula_v1.pkl')
    dla_df = get_pickle('../../cache/dla_v1.pkl')
    new_recovery = get_pickle('../../cache/new_recovery_v1.pkl')
    print('ULA, DLA, New recovery loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print("[PROGRESS] Data Fetch Complete")


Model scores fetched: 108,924 records
ULA ready
DLA ready
New recovery ready
ULA records: 1,683,196
[PROGRESS] Data Fetch Complete


In [6]:
# =============================================================================
# CELL 6: SHARED DATA PREP (dates, period columns, filters)
# =============================================================================
# Period assignment and date-range filtering are deferred to the
# per-granularity loop in Cell 8.

# Filter out Core LOB
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

# Ensure date columns are proper types
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# Pre-compute all period columns so the loop can pick whichever it needs
for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')

# Convert week columns to str for compatibility
for df in [ula_df_total, new_recovery]:
    df['book_week'] = df['book_week'].astype(str)
    df['app_week'] = df['app_week'].astype(str)

# String version of date_col for flag comparisons
ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)

print(f"Shared data prep complete: {len(ula_df_total):,} ULA rows, {len(new_recovery):,} recovery rows")

Shared data prep complete: 1,683,196 ULA rows, 1,240,970 recovery rows


In [7]:
# =============================================================================
# CELL 7: FLAG CREATION AND DATA REFINEMENT
# =============================================================================

date_col_str = f'{date_col}_str'

# --- ULA Processing ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- DLA Merge ---
dla_df = dla_df.rename(columns={"valid_vintage": "book_vintage"})
dla_explicit = dla_df[dla_df['book_vintage'] != 'current']
dla_current = dla_df[dla_df['book_vintage'] == 'current'].drop(columns='book_vintage')
last_explicit_vintage = dla_explicit['book_vintage'].max()

ula_df_total = ula_df_total.merge(dla_explicit, how='left', on=['dealer_number', 'book_vintage'])
new_and_missing = ula_df_total['pricing_scalar'].isna() & (ula_df_total['book_vintage'] > last_explicit_vintage)
fallback = ula_df_total.loc[new_and_missing, ['dealer_number']].merge(dla_current, on='dealer_number', how='left')
for col in ['dll_edition', 'loss_ratio', 'dealer_level', 'pricing_scalar']:
    ula_df_total.loc[new_and_missing, col] = fallback[col].values

ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)
ula_df_total.loc[ula_df_total.frni_flag == 1, 'pricing_scalar'] *= 1.05
ula_df_total.loc[ula_df_total.frni_flag == 0, 'pricing_scalar'] *= 0.95

# --- Driver Flag (derived from ULA job_company) ---
warnings.filterwarnings("ignore", category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('not provided')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

# --- ULA NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- Weekly-matching filters (aligned with weekly.ipynb) ---
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[ula_df_total.amt_financed <= 75000]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[
    (ula_df_total.lob == 'MCY') |
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

print(f"ULA after weekly-matching filters: {len(ula_df_total):,}")

# --- NonKMX Flags ---
ula_df_total['ent_fld_flag'] = (ula_df_total.lob == 'ENT') | (ula_df_total.lob == 'FLD')
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = (ula_df_total.lob == 'ENT')
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & ula_df_total.lob.isin({'AN', 'FLD', 'FRN', 'STG'}) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = np.where(ula_df_total.student_loan_flag == 1, 1, 0)

# --- KMX Flags ---
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = ula_df_total.kmx_npc_flag == 1
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA', 'FL', 'CO'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['illinois_flag'] = ula_df_total.state == 'IL'
ula_df_total['mississippi_flag'] = ula_df_total.state == 'MS'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_toyho_flag'] = (ula_df_total.cd_model_score >= 130) & ula_df_total.make.isin({'HON', 'TOY'})
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- MTN 4.1 model score transformation (applied to ULA source) ---
is_mtn41_ula = ula_df_total.mtn_model == 4.1
ula_df_total.loc[is_mtn41_ula, 'cd_model_score'] = (
    142 + (ula_df_total.loc[is_mtn41_ula, 'cd_model_score'] - 142) * 1.5
)

# Vintage formatting and model-score aggregation are deferred to the
# per-granularity loop in Cell 8.

print(f"ULA after refinement: {len(ula_df_total):,}")

ULA after weekly-matching filters: 1,677,031
ULA after refinement: 130,852


In [8]:
# =============================================================================
# CELL 8: MULTI-GRANULARITY PIPELINE
# Loops over each granularity: period assignment -> vintage -> ms_df ->
# RAGU scoring -> rollup -> Excel export.
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def get_ragu_score(vintage, lob, ula_data, nr_data, ms_data, baseline_config, leave_out='None'):
    """Core RAGU Score calculation for a single vintage and individual LOB."""
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']
    ltv_mult = 17 / 0.65 if lob == 'KMX' else 17
    apr_mult = 0.7 / 0.65 if lob == 'KMX' else 0.7

    ula_df = ula_data[(ula_data.vintage == vintage) & (ula_data.lob == lob)].copy()

    if len(ula_df) == 0:
        return None

    if lob == 'KMX':
        ula_df = get_ula_multiplier_kmx(ula_df, leave_out=leave_out)
    else:
        ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out)

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    nr = nr_data[['account_number', 'new_recovery_multiplier']].drop_duplicates(subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')

    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    full_pop_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'ltv', 'bbvalue', 'apr'],
        include_groups=False
    )

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()]
    recovery_df = recovery_df.copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['recovery_unadjusted_multiplier'],
        include_groups=False
    )
    recovery_metrics = recovery_metrics.drop(columns='amt_financed')

    grouped_mix_df = full_pop_metrics.join(recovery_metrics)

    vintage_ms_df = ms_data[ms_data['period'] == vintage].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')

    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * ltv_mult
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * apr_mult
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    full_df['vintage'] = vintage
    return full_df


# --- Constants for rollup and Excel export ---
PERIOD_KEY = {'q': 'quarter', 'm': 'month', 'w': 'week'}
EXCEL_SHEET_MAP = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}
EXCEL_OUTPUT = '../output/barebones_ragu.xlsx'

ROLLUP_METRICS = [
    'ms_original', 'gross_loss_impact', 'recovery_impact',
    'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr',
]

METRIC_ROWS = [
    ('Model Score',                'ms_original'),
    ('Expected Gross Loss Impact', 'gross_loss_impact'),
    ('Recovery Impact',            'recovery_impact'),
    ('LTV Impact',        'ltv_impact'),
    ('APR Impact',        'apr_impact'),
    ('RAGU Score',        'ragu_score'),
    ('Amount Financed',   'amt_financed_x'),
    ('Weighted LTV',      'ltv'),
    ('Weighted APR',      'apr'),
]


# --- Main multi-granularity loop ---
for g in granularities:
    print(f"\n{'='*60}")
    print(f"Processing granularity: {g}")
    print(f"{'='*60}")

    period_freq = PERIOD_FREQ_MAP[g]
    g_start_period = start_date.to_period(period_freq)
    g_end_period = end_date.to_period(period_freq)

    # 1. Copy shared data and assign period / filter date range
    ula_g = ula_df_total.copy()
    nr_g = new_recovery.copy()

    for df in [ula_g, nr_g]:
        df['period'] = df[PERIOD_KEY[g]]
        mask = (df['period'] >= g_start_period) & (df['period'] <= g_end_period)
        df.drop(df[~mask].index, inplace=True)

    # 2. Format vintage labels
    ula_g['vintage'] = format_vintage(ula_g['period'])
    nr_g['vintage'] = format_vintage(nr_g['period'])

    # 3. Aggregate model scores
    ms_df = ula_g.groupby(['period', 'lob']).apply(
        weighted_average_and_sum, 'cd_model_score', include_groups=False
    ).reset_index()
    ms_df = ms_df.rename(columns={'cd_model_score': 'model_score'})
    ms_df['period'] = format_vintage(ms_df['period'])

    print(f"  Periods: {ula_g['period'].nunique()} | "
          f"Range: {ula_g['period'].min()} to {ula_g['period'].max()} | "
          f"MS combos: {len(ms_df)}")

    # 4. RAGU scoring - individual LOBs
    all_vintages = sorted(ula_g['vintage'].unique())
    results = []

    for lob in LOBS:
        excluded = EXCLUDED_VINTAGES.get(lob, set())
        baseline_config = BASELINES[lob]

        for vintage in all_vintages:
            if vintage in excluded:
                continue
            try:
                result = get_ragu_score(vintage, lob, ula_g, nr_g, ms_df, baseline_config)
                if result is not None:
                    results.append(result)
            except Exception as e:
                print(f"  Error: {vintage} {lob}: {e}")

        print(f"  {lob} complete")

    all_df = pd.concat(results, ignore_index=False).reset_index()
    print(f"  Scoring: {len(all_df)} rows across {all_df.vintage.nunique()} vintages")

    # 5. Rollup aggregation
    for group_name, group_lobs in ROLLUP_GROUPS.items():
        group_df = all_df[all_df.lob.isin(group_lobs)].copy()
        group_df = group_df.rename(columns={'amt_financed_x': 'amt_financed'})
        rollup = group_df.groupby('vintage').apply(
            weighted_average_and_sum, ROLLUP_METRICS, include_groups=False
        ).reset_index()
        rollup['lob'] = group_name
        rollup = rollup.rename(columns={'amt_financed': 'amt_financed_x'})
        all_df = pd.concat([all_df, rollup], ignore_index=True)

    print(f"  After rollups: {len(all_df)} rows across {all_df.lob.nunique()} groups")

    # 6. Excel export
    sheet_name = EXCEL_SHEET_MAP[g]
    sorted_vintages = sorted(all_df['vintage'].unique())

    if os.path.exists(EXCEL_OUTPUT):
        wb = openpyxl.load_workbook(EXCEL_OUTPUT)
        if sheet_name in wb.sheetnames:
            del wb[sheet_name]
        ws = wb.create_sheet(sheet_name)
    else:
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = sheet_name

    current_row = 1
    all_export_lobs = LOBS + list(ROLLUP_GROUPS.keys())

    for lob in all_export_lobs:
        lob_data = all_df[all_df.lob == lob].set_index('vintage')

        ws.cell(row=current_row, column=1, value=lob)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            ws.cell(row=current_row, column=col_idx, value=v)
        current_row += 1

        for label, col_key in METRIC_ROWS:
            ws.cell(row=current_row, column=1, value=label)
            for col_idx, v in enumerate(sorted_vintages, start=2):
                if v in lob_data.index:
                    ws.cell(row=current_row, column=col_idx, value=lob_data.loc[v, col_key])
            current_row += 1

        current_row += 1

    wb.save(EXCEL_OUTPUT)
    print(f"  Saved to {EXCEL_OUTPUT} (sheet: {sheet_name})")
    print(f"  {len(all_export_lobs)} groups x {len(sorted_vintages)} periods")

# Preserve globals for downstream cells (set to last processed granularity)
granularity = granularities[-1]
period_freq = PERIOD_FREQ_MAP[granularity]
start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)
print(f"\n[PROGRESS] All granularities complete: {granularities}")



Processing granularity: w
  Periods: 37 | Range: 2025-12-28/2026-01-03 to 2026-09-06/2026-09-12 | MS combos: 332
  AN complete
  FRN complete
  STG complete
  FLD complete
  Error: 2026-09-06/2026-09-12 ENT: columns overlap but no suffix specified: Index(['loss_multiplier', 'ltv', 'bbvalue', 'apr'], dtype='str')
  ENT complete
  KMX complete
  Scoring: 221 rows across 37 vintages
  After rollups: 332 rows across 9 groups
  Saved to ../output/barebones_ragu.xlsx (sheet: Data Tables (W))
  9 groups x 37 periods

[PROGRESS] All granularities complete: ['w']


In [9]:
# Rollup logic has been absorbed into the multi-granularity loop in Cell 8 above.
pass

In [10]:
# Excel export logic has been absorbed into the multi-granularity loop in Cell 8 above.
pass


In [11]:
# =============================================================================
# CELL 10: CSV OUTPUT AND DIAGNOSTICS
# =============================================================================
# When running multiple granularities, CSV/sandbox export is skipped --
# only Excel sheets are populated (per the multi-granularity workflow).


run_sandbox = True
run_historical = True

if len(granularities) > 1:
    print(f"Multi-granularity run ({granularities}): CSV and sandbox export skipped.")
    print("Excel sheets have already been populated in Cell 8.")
    run_sandbox = False
    run_historical = False
else:
    col_rename = {
        'ms_original': 'model_score',
        'gross_loss_impact': 'gross_loss',
        'recovery_impact': 'recovery',
        'ltv_impact': 'ltv',
        'apr_impact': 'apr',
    }
    final_cols = ['lob', 'vintage', 'model_score', 'gross_loss', 'recovery', 'ltv', 'apr', 'ragu_score', 'amt_financed_x']
    all_df['month_run'] = pd.Timestamp.now().strftime('%Y M%m')

    if 'ms_original' in all_df.columns:
        source_cols = ['lob', 'vintage', 'ms_original', 'gross_loss_impact', 'recovery_impact', 'ltv_impact', 'apr_impact', 'ragu_score', 'amt_financed_x']
        output_df = all_df[source_cols + ['month_run']].rename(columns=col_rename)
    else:
        output_df = all_df[final_cols + ['month_run']].copy()

    output_df.to_csv('../output/all_df.csv', index=False)
    print(f"Saved all_df.csv ({len(output_df)} rows)")

    display(output_df.drop(columns='month_run').head(20))

# --- Redshift Upload ---

def upload_ragu_to_redshift(df, table='sandbox.ragu_monthend_current'):
    """Upload a DataFrame to a Redshift table via INSERT INTO VALUES.
    Drops and recreates the table each run for a clean refresh."""
    upload_df = df.copy().reset_index(drop=True)
    upload_df['insert_column'] = (
        "('" + upload_df['lob'].astype(str)
        + "', '" + upload_df['vintage'].astype(str)
        + "', " + upload_df['model_score'].round(4).astype(str)
        + ", " + upload_df['gross_loss'].round(4).astype(str)
        + ", " + upload_df['recovery'].round(4).astype(str)
        + ", " + upload_df['ltv'].round(4).astype(str)
        + ", " + upload_df['apr'].round(4).astype(str)
        + ", " + upload_df['ragu_score'].round(4).astype(str)
        + ", '" + upload_df['month_run'].astype(str)
        + "')"
    )
    values_str = upload_df['insert_column'].str.cat(sep=',').replace("'nan'", 'null')

    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"DROP TABLE IF EXISTS {table};")
        cur.execute(f"""
            CREATE TABLE {table} (
                lob         VARCHAR(25),
                vintage     VARCHAR(20),
                model_score FLOAT,
                gross_loss  FLOAT,
                recovery    FLOAT,
                ltv         FLOAT,
                apr         FLOAT,
                ragu_score  FLOAT,
                month_run   VARCHAR(10)
            );
        """)
        cur.execute(f"INSERT INTO {table} VALUES {values_str}")
        conn.commit()
    print(f"Uploaded {len(upload_df)} rows to {table}")


def upload_ragu_historical(month_run_val, table='sandbox.ragu_monthend',
                           source='sandbox.ragu_monthend_current'):
    """Append current-table rows into historical with current_version_flag.
    Idempotent: deletes any existing rows for this month_run before inserting."""
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"""
            CREATE TABLE IF NOT EXISTS {table} (
                lob                  VARCHAR(25),
                vintage              VARCHAR(20),
                model_score          FLOAT,
                gross_loss           FLOAT,
                recovery             FLOAT,
                ltv                  FLOAT,
                apr                  FLOAT,
                ragu_score           FLOAT,
                month_run            VARCHAR(10),
                current_version_flag SMALLINT DEFAULT 0
            );
        """)

        cur.execute(f"DELETE FROM {table} WHERE month_run = '{month_run_val}'")

        cur.execute(f"UPDATE {table} SET current_version_flag = 0 WHERE current_version_flag = 1")

        cur.execute(f"INSERT INTO {table} SELECT *, 1 FROM {source}")
        conn.commit()
    print(f"Historical table {table} updated (month_run={month_run_val}, flag=1)")

if run_sandbox:
    if granularities[0] != 'm':
        print(f"Sandbox upload skipped (granularity='{granularities[0]}'). Sandbox upload only runs for monthly ('m') granularity.")
    else:
        month_run_val = output_df['month_run'].iloc[0]
        sandbox_df = output_df[output_df['vintage'] < month_run_val].drop(columns='amt_financed_x')
        print(f"Filtered to {len(sandbox_df)} rows (excluded vintages >= {month_run_val})")
        upload_ragu_to_redshift(sandbox_df)

if run_historical:
    upload_ragu_historical(output_df['month_run'].iloc[0])
print("[PROGRESS] Export Complete")


Saved all_df.csv (332 rows)


,lob,vintage,model_score,gross_loss,recovery,ltv,apr,ragu_score,amt_financed_x
0,AN,2025-12-28/2026-01-03,140.489577,3.352921,4.707175,3.709554,0.908655,153.167882,3220104.85
1,AN,2026-01-04/2026-01-10,140.531838,3.543627,3.671818,3.568928,0.632101,151.948311,2980427.93
2,AN,2026-01-11/2026-01-17,140.428479,3.030213,3.086902,4.472700,0.532795,151.551089,2913291.58
3,AN,2026-01-18/2026-01-24,140.133273,3.605848,1.985700,2.955830,0.158302,148.838952,2476521.89
4,AN,2026-01-25/2026-01-31,140.423006,2.817464,2.531648,5.023128,1.041746,151.836992,3187504.58
5,AN,2026-02-01/2026-02-07,139.657795,2.968754,1.931605,4.078892,-0.104026,148.533020,3487655.37
6,AN,2026-02-08/2026-02-14,140.716844,3.003321,1.629399,4.172500,0.481342,150.003407,3939777.66
7,AN,2026-02-15/2026-02-21,140.266130,3.107341,1.062412,4.725888,0.803616,149.965388,5484726.75
8,AN,2026-02-22/2026-02-28,139.505652,2.213558,1.160679,5.254184,0.206235,148.340308,7653936.42
9,AN,2026-03-01/2026-03-07,141.369886,2.673951,2.339470,4.243098,0.753679,151.380084,5920748.52


Sandbox upload skipped (granularity='w'). Sandbox upload only runs for monthly ('m') granularity.
Historical table sandbox.ragu_monthend updated (month_run=2026 M09, flag=1)
[PROGRESS] Export Complete


In [12]:
current_quarter = ula_df_total['book_vintage'].max()
current_quarter

'2026 Q3'